In [ ]:
# ── セル1: UI パネル（パラメータ調整） ────────────────────

!pip install -q ipywidgets nest-asyncio edge-tts

from ipywidgets import IntSlider, Textarea, Dropdown, Button, VBox, HBox, Label, Output
from IPython.display import display, clear_output, Audio
import ipywidgets as widgets
import nest_asyncio
import asyncio
import datetime
import os
import edge_tts

nest_asyncio.apply()

LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

VOICE_PRIORITY = [
    "ja-JP-NanamiNeural",
    "ja-JP-KeitaNeural",
]

# 最後に生成されたファイルパスを保持
last_generated_audio = None

VOICE_PRIORITY = globals().get("VOICE_PRIORITY", VOICE_PRIORITY)

text_input = Textarea(
    value="こんにちは。今日はどんなお話をしましょうか？もしよければ、ゆっくりお話ししますね。",
    placeholder="読み上げるテキストを入力してください",
    description="テキスト:",
    rows=4,
    style={'description_width': '80px'}
)

voice_dropdown = Dropdown(
    options=VOICE_PRIORITY,
    value=VOICE_PRIORITY[0],
    description="音声:",
    style={'description_width': '80px'}
)

rate_slider = IntSlider(
    value=-5, min=-50, max=50, step=1,
    description="速度（Rate）:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
rate_label = Label(value="-5%")

pitch_slider = IntSlider(
    value=10, min=-50, max=50, step=1,
    description="ピッチ:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
pitch_label = Label(value="+10Hz")

volume_slider = IntSlider(
    value=0, min=-100, max=100, step=1,
    description="ボリューム:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
volume_label = Label(value="+0%")

def update_labels(change):
    rate_label.value   = f"{rate_slider.value:+d}%"
    pitch_label.value  = f"{pitch_slider.value:+d}Hz"
    volume_label.value = f"{volume_slider.value:+d}%"

rate_slider.observe(update_labels, 'value')
pitch_slider.observe(update_labels, 'value')
volume_slider.observe(update_labels, 'value')

execute_button = Button(
    description="🎵 実行（合成・再生）",
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
output_area = Output()

async def synthesize_with_ui():
    global last_generated_audio
    text      = text_input.value
    voice     = voice_dropdown.value
    rate_val  = rate_slider.value
    pitch_val = pitch_slider.value
    vol_val   = volume_slider.value

    output_area.clear_output()
    with output_area:
        ts          = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"{LOCAL_AUDIO_DIR}/edge_{ts}.mp3"
        print(f"⏳ 処理中...")
        print(f"  テキスト: {text[:40]}...")
        print(f"  音声: {voice}")
        print(f"  Rate: {rate_val:+d}%, Pitch: {pitch_val:+d}Hz, Volume: {vol_val:+d}%")
        try:
            params = {}
            if rate_val  != 0: params['rate']   = f"{rate_val:+d}%"
            if pitch_val != 0: params['pitch']  = f"{pitch_val:+d}Hz"
            if vol_val   != 0: params['volume'] = f"{vol_val:+d}%"
            communicate = edge_tts.Communicate(text, voice=voice, **params)
            await communicate.save(output_path)
            last_generated_audio = output_path
            print(f"\n✅ 完了！")
            print(f"📁 保存: {output_path}")
            print(f"💡 セル2の後工程パネルで音声加工が可能です")
            display(Audio(output_path, autoplay=False))
        except Exception as e:
            print(f"\n❌ エラー: {type(e).__name__}: {e}")

def on_execute_clicked(button):
    asyncio.run(synthesize_with_ui())

execute_button.on_click(on_execute_clicked)

settings_box = VBox([
    Label(value="⚙️ 音声パラメータ調整", style={'font_weight': 'bold'}),
    HBox([rate_slider,   rate_label]),
    HBox([pitch_slider,  pitch_label]),
    HBox([volume_slider, volume_label]),
])
input_box = VBox([
    Label(value="📋 入力設定", style={'font_weight': 'bold'}),
    HBox([Label(value="テキスト:"), text_input]),
    HBox([Label(value="音声:"),     voice_dropdown]),
])
control_panel = VBox([
    input_box, settings_box, execute_button, output_area
], layout=widgets.Layout(border='1px solid #ddd', padding='15px', border_radius='5px'))

display(control_panel)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🎛️  セル2 ── 後工程パネル v2（高品質・選択的加工）
#
# 改善点:
#   - フォルマントシフト → リサンプリング方式に変更（音質向上）
#   - イントネーション → 有声区間のみ・滑らかなブレンド適用
#   - ロボット → 位相ボコーダー方式（自然な固定ピッチ）
#   - 全加工後にピーク正規化で音割れ防止
# ══════════════════════════════════════════════════════════════

!pip install -q librosa soundfile pydub scipy soxr

import librosa
import librosa.effects
import soundfile as sf
import numpy as np
from scipy import signal as scipy_signal
from scipy.interpolate import interp1d
from pydub import AudioSegment
import ipywidgets as widgets
from ipywidgets import (
    FloatSlider, IntSlider, Dropdown, Button,
    VBox, HBox, Label, Output, HTML, FloatRangeSlider, Checkbox
)
from IPython.display import display, Audio
import datetime, os, warnings
warnings.filterwarnings('ignore')

LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

# ══════════════════════════════════════════════
# 🎭 プリセット定義
# ══════════════════════════════════════════════
PRESETS = {
    "── プリセット選択 ──":  {"pitch": 0.0,  "speed": 1.00, "formant": 0.0,  "robot": 0.0},
    "👦 子供（男の子）":      {"pitch": +5.0, "speed": 1.10, "formant": +2.5, "robot": 0.0},
    "👧 子供（女の子）":      {"pitch": +7.0, "speed": 1.12, "formant": +3.0, "robot": 0.0},
    "🧑 大人（男性）":        {"pitch": -3.0, "speed": 0.97, "formant": -2.0, "robot": 0.0},
    "👩 大人（女性）":        {"pitch": +2.0, "speed": 1.00, "formant": +1.0, "robot": 0.0},
    "🧓 老人（男性）":        {"pitch": -4.0, "speed": 0.88, "formant": -2.5, "robot": 0.0},
    "👵 老人（女性）":        {"pitch": -1.5, "speed": 0.90, "formant": -1.0, "robot": 0.0},
    "🤖 ロボット風":          {"pitch":  0.0, "speed": 1.00, "formant":  0.0, "robot": 0.75},
    "✏️ 手動設定":            None,
}

# ──────────────────────────────────────────────
# 🔧 音声加工コアロジック（v2）
# ──────────────────────────────────────────────

def load_audio(path: str):
    """MP3/WAV → float32 numpy + sr（モノラル）"""
    audio = AudioSegment.from_file(path).set_channels(1)
    sr    = audio.frame_rate
    y     = np.array(audio.get_array_of_samples(), dtype=np.float32)
    y    /= 2 ** (audio.sample_width * 8 - 1)
    return y, sr

def pitch_shift_clean(y, sr, n_steps: float):
    """librosa pitch_shift（音質優先・STFT使用）"""
    if abs(n_steps) < 0.01:
        return y
    return librosa.effects.pitch_shift(
        y, sr=sr, n_steps=n_steps,
        res_type='soxr_hq'  # soxr高品質（resampy不要）
    )

def time_stretch_clean(y, rate: float):
    """librosa time_stretch（位相ボコーダー・音質優先）"""
    if abs(rate - 1.0) < 0.01:
        return y
    return librosa.effects.time_stretch(y, rate=rate)

def formant_shift_resample(y, sr, semitones: float):
    """
    フォルマントシフト（リサンプリング方式）:
    - 速度変換でフォルマントを動かした後、
      時間軸を元に戻すことで長さを維持しながら声質を変える
    - 波形を直接操作しないため音質劣化が少ない
    """
    if abs(semitones) < 0.01:
        return y
    ratio     = 2 ** (semitones / 12.0)   # e.g. +3半音 → ×1.189
    n_target  = int(len(y) / ratio)        # リサンプル後サンプル数
    # 高品質リサンプリングでフォルマントを移動
    y_resampled = librosa.resample(y, orig_sr=sr, target_sr=int(sr * ratio),
                                   res_type='soxr_hq')
    # 元の長さに戻す（ピッチは変わらず声質だけ変わる）
    y_stretched = librosa.effects.time_stretch(
        y_resampled, rate=float(len(y_resampled)) / len(y)
    )
    # 長さを元に合わせる
    if len(y_stretched) >= len(y):
        return y_stretched[:len(y)]
    return np.pad(y_stretched, (0, len(y) - len(y_stretched)))

def robot_vocoder(y, sr, strength: float):
    """
    ロボット風エフェクト（STFTマグニチュード再合成）:
    - 位相情報を破棄し、マグニチュードのみで再合成
    - strength でオリジナルとブレンド
    """
    if strength < 0.01:
        return y
    D      = librosa.stft(y, n_fft=2048, hop_length=512)
    mag    = np.abs(D)
    # 位相をゼロ（コサイン成分のみ）で再合成
    D_robot = mag.astype(complex)           # 位相=0（実数のみ）
    y_robot = librosa.istft(D_robot, length=len(y), hop_length=512)
    return (1.0 - strength) * y + strength * y_robot

# ───────────────────────────────────────────────────────────
# 🎭 イントネーション加工（選択的・有声区間ブレンド方式）
#
# 従来との違い:
#   - 有声区間（voiced）のみを対象にする
#   - 各フレームでピッチシフト量を計算し、元波形とソフトブレンド
#   - 無声音・休止部は一切触らない → ドラム缶感を排除
#   - 時間範囲（%指定）で部分適用が可能
# ───────────────────────────────────────────────────────────

def intonation_selective(
    y: np.ndarray,
    sr: int,
    strength: float,          # >1 で強調、<1 でフラット
    time_range: tuple,        # (start_pct, end_pct) 0〜100
    blend: float = 0.6,       # 加工済みとオリジナルのブレンド率
):
    """
    選択的イントネーション強調
    - time_range: (0, 100) なら全体、(25, 75) なら中間50%のみ加工
    - blend: 0=オリジナルのまま, 1=完全加工（デフォルト0.6で自然に）
    """
    if abs(strength - 1.0) < 0.05:
        return y

    hop_length = 256   # 細かめのホップで精度向上
    n_fft      = 2048

    # f0 推定（pyin: 安定した推定アルゴリズム）
    f0, voiced_flag, voiced_prob = librosa.pyin(
        y, sr=sr,
        fmin=librosa.note_to_hz('C2'),
        fmax=librosa.note_to_hz('C7'),
        hop_length=hop_length,
        fill_na=None,
    )

    if f0 is None or np.all(np.isnan(f0)):
        print("  ⚠️ ピッチ推定失敗: イントネーション加工をスキップ")
        return y

    # 有声フレームのf0から中央値（基準ピッチ）を計算
    voiced_f0 = f0[voiced_flag]
    if len(voiced_f0) == 0:
        return y
    median_f0 = np.nanmedian(voiced_f0)

    # 時間範囲をサンプルインデックスに変換
    total_samples = len(y)
    range_start   = int(total_samples * time_range[0] / 100)
    range_end     = int(total_samples * time_range[1] / 100)

    # 出力バッファ（オリジナルコピー）
    output = y.copy()

    # フレーム単位で有声区間のみ加工
    n_frames = len(f0)
    for i in range(n_frames):
        # 無声区間はスキップ
        if not voiced_flag[i]:
            continue
        f = f0[i]
        if f is None or np.isnan(f) or f <= 0:
            continue

        # このフレームのサンプル範囲
        s_start = i * hop_length
        s_end   = min(s_start + hop_length * 4, total_samples)

        # 時間範囲外はスキップ
        if s_end < range_start or s_start > range_end:
            continue

        # 範囲端でのフェードイン・アウト（境界の不自然さを軽減）
        local_blend = blend
        margin = hop_length * 4  # フェード幅
        if s_start < range_start + margin:
            local_blend *= (s_start - range_start) / margin
        elif s_end > range_end - margin:
            local_blend *= (range_end - s_end) / margin
        local_blend = max(0.0, min(blend, local_blend))

        # 現在のf0から中央値への偏差（半音単位）
        if median_f0 <= 0 or f <= 0:
            continue
        deviation_st = 12 * np.log2(f / median_f0)
        # 偏差を strength 倍に拡大（強調）or 縮小（フラット化）
        target_st    = deviation_st * strength
        delta_st     = target_st - deviation_st

        if abs(delta_st) < 0.1:  # 微小変化はスキップ
            continue

        chunk = y[s_start:s_end]
        if len(chunk) < 128:
            continue

        # チャンク単位でピッチシフト
        shifted = librosa.effects.pitch_shift(
            chunk, sr=sr, n_steps=delta_st,
            res_type='soxr_hq'  # soxr高品質（resampy不要）
        )
        shifted = shifted[:len(chunk)]

        # オリジナルとブレンド（ハードな切り替えを避ける）
        output[s_start:s_start + len(shifted)] = (
            (1.0 - local_blend) * chunk[:len(shifted)] +
            local_blend          * shifted
        )

    return output

# ══════════════════════════════════════════════
# 🎚️ UI ウィジェット
# ══════════════════════════════════════════════

# プリセット
preset_dd = Dropdown(
    options=list(PRESETS.keys()),
    value="── プリセット選択 ──",
    description="プリセット:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)

# ピッチシフト
pitch_sl = FloatSlider(
    value=0.0, min=-12.0, max=12.0, step=0.5,
    description="ピッチ(半音):",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.1f'
)
pitch_lbl = Label(value="0.0 半音")

# 速度
speed_sl = FloatSlider(
    value=1.0, min=0.5, max=2.0, step=0.05,
    description="速度(倍率):",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.2f'
)
speed_lbl = Label(value="×1.00")

# フォルマント
formant_sl = FloatSlider(
    value=0.0, min=-6.0, max=6.0, step=0.5,
    description="声質(フォルマント):",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.1f'
)
formant_lbl = Label(value="0.0（中性）")

# ロボット
robot_sl = FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.05,
    description="ロボット強度:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.2f'
)
robot_lbl = Label(value="0.00（オフ）")

# ─── イントネーションセクション ───
intn_enable = Checkbox(
    value=False,
    description="イントネーション加工を有効にする",
    style={'description_width': '240px'}
)

intn_strength_sl = FloatSlider(
    value=1.5, min=0.2, max=3.0, step=0.1,
    description="抑揚強度:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.1f'
)
intn_strength_lbl = Label(value="×1.5（強調）")

intn_range_sl = FloatRangeSlider(
    value=[0, 100], min=0, max=100, step=5,
    description="適用範囲(%):",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.0f'
)
intn_range_lbl = Label(value="全体(0〜100%)")

intn_blend_sl = FloatSlider(
    value=0.6, min=0.1, max=1.0, step=0.05,
    description="ブレンド量:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.2f'
)
intn_blend_lbl = Label(value="0.60（自然）")

# イントネーション詳細を折りたたみ
intn_detail_box = VBox([
    HBox([intn_strength_sl, intn_strength_lbl]),
    HBox([intn_range_sl,    intn_range_lbl]),
    HBox([intn_blend_sl,    intn_blend_lbl]),
], layout=widgets.Layout(display='none', padding='5px 0 0 20px'))

def on_intn_enable(change):
    intn_detail_box.layout.display = '' if change['new'] else 'none'
intn_enable.observe(on_intn_enable, names='value')

# ラベル更新
def update_labels(change):
    pitch_lbl.value   = f"{pitch_sl.value:+.1f} 半音"
    speed_lbl.value   = f"×{speed_sl.value:.2f}"
    fv = formant_sl.value
    formant_lbl.value = f"{fv:+.1f}（{'低め' if fv < -0.5 else ('高め' if fv > 0.5 else '中性')}）"
    robot_lbl.value   = f"{robot_sl.value:.2f}（{'オン' if robot_sl.value > 0 else 'オフ'}）"
    iv = intn_strength_sl.value
    intn_strength_lbl.value = f"×{iv:.1f}（{'フラット化' if iv < 0.8 else ('強調' if iv > 1.2 else '標準')}）"
    r = intn_range_sl.value
    intn_range_lbl.value = "全体" if (r[0] == 0 and r[1] == 100) else f"{r[0]:.0f}〜{r[1]:.0f}%"
    intn_blend_lbl.value = f"{intn_blend_sl.value:.2f}（{'ソフト' if intn_blend_sl.value < 0.4 else ('強め' if intn_blend_sl.value > 0.8 else '自然')}）"

for w in [pitch_sl, speed_sl, formant_sl, robot_sl,
          intn_strength_sl, intn_range_sl, intn_blend_sl]:
    w.observe(update_labels, 'value')

# プリセット選択時のスライダー自動セット
def on_preset(change):
    p = PRESETS.get(change['new'])
    if p is None:   # 手動
        return
    pitch_sl.value   = p['pitch']
    speed_sl.value   = p['speed']
    formant_sl.value = p['formant']
    robot_sl.value   = p['robot']
preset_dd.observe(on_preset, names='value')

# 入力ファイル
source_dd = Dropdown(
    options=["セル1の最新ファイルを使用", "ファイルパスを直接指定"],
    value="セル1の最新ファイルを使用",
    description="入力元:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
manual_path = widgets.Text(
    placeholder="/content/audio_output/edge_XXXXXXXX.mp3",
    description="ファイルパス:",
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)
manual_path_box = VBox([manual_path], layout=widgets.Layout(display='none'))
def on_source(change):
    manual_path_box.layout.display = '' if change['new'] == "ファイルパスを直接指定" else 'none'
source_dd.observe(on_source, names='value')

# ══════════════════════════════════════════════
# 🔘 実行ボタン
# ══════════════════════════════════════════════
process_btn = Button(
    description="🔊 加工して再生",
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
proc_out = Output()

def on_process(btn):
    proc_out.clear_output()
    with proc_out:
        # 入力ファイル特定
        if source_dd.value == "セル1の最新ファイルを使用":
            src = globals().get('last_generated_audio')
            if not src or not os.path.exists(src):
                files = sorted([
                    os.path.join(LOCAL_AUDIO_DIR, f)
                    for f in os.listdir(LOCAL_AUDIO_DIR)
                    if f.endswith('.mp3') and 'processed' not in f
                ])
                src = files[-1] if files else None
        else:
            src = manual_path.value.strip()

        if not src or not os.path.exists(src):
            print("❌ 入力ファイルが見つかりません")
            return

        print(f"📂 入力: {os.path.basename(src)}")
        print("⏳ 加工中...\n")

        try:
            y, sr = load_audio(src)
            print(f"  ✔ 読込完了 (sr={sr}Hz, {len(y)/sr:.1f}秒)")

            # ── 加工パイプライン ──────────────────────

            # 1. ロボット（最初に適用）
            if robot_sl.value > 0.01:
                print(f"  🤖 ロボット (強度={robot_sl.value:.2f})")
                y = robot_vocoder(y, sr, robot_sl.value)

            # 2. 速度変更（タイムストレッチ）
            if abs(speed_sl.value - 1.0) > 0.01:
                print(f"  ⏩ 速度 (×{speed_sl.value:.2f})")
                y = time_stretch_clean(y, rate=speed_sl.value)

            # 3. ピッチシフト
            if abs(pitch_sl.value) > 0.01:
                print(f"  🎵 ピッチ ({pitch_sl.value:+.1f}半音)")
                y = pitch_shift_clean(y, sr, pitch_sl.value)

            # 4. フォルマントシフト（リサンプリング方式）
            if abs(formant_sl.value) > 0.01:
                print(f"  🗣️ フォルマント ({formant_sl.value:+.1f}半音相当)")
                y = formant_shift_resample(y, sr, formant_sl.value)

            # 5. イントネーション（有効な場合のみ・選択的）
            if intn_enable.value:
                r = intn_range_sl.value
                print(f"  🎭 イントネーション (×{intn_strength_sl.value:.1f},"
                      f" 範囲={r[0]:.0f}〜{r[1]:.0f}%,"
                      f" ブレンド={intn_blend_sl.value:.2f})")
                y = intonation_selective(
                    y, sr,
                    strength   = intn_strength_sl.value,
                    time_range = (r[0], r[1]),
                    blend      = intn_blend_sl.value,
                )

            # ── 正規化 ───────────────────────────────
            peak = np.max(np.abs(y))
            if peak > 0:
                y = y / peak * 0.95

            # ── 保存 ────────────────────────────────
            ts      = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            preset  = preset_dd.value.split(' ')[0] if preset_dd.value != "── プリセット選択 ──" else "custom"
            out_path = f"{LOCAL_AUDIO_DIR}/processed_{ts}.wav"
            sf.write(out_path, y, sr, subtype='PCM_16')

            print(f"\n✅ 完了！")
            print(f"📁 保存: {out_path}")
            display(Audio(out_path, autoplay=True))

        except Exception as e:
            import traceback
            print(f"\n❌ エラー: {type(e).__name__}: {e}")
            traceback.print_exc()

process_btn.on_click(on_process)

# ══════════════════════════════════════════════
# 🎨 レイアウト
# ══════════════════════════════════════════════
header = HTML(value="""
<div style='
    background: linear-gradient(135deg, #1a1a2e 0%, #0f3460 100%);
    color: #e8eaf6;
    padding: 12px 20px;
    border-radius: 6px 6px 0 0;
    font-family: "Courier New", monospace;
    font-size: 14px;
    letter-spacing: 1.5px;
'>
🎛️ &nbsp; 後工程パネル v2 &nbsp;—&nbsp; 高品質・選択的音声加工
</div>
""")

intn_section = VBox([
    Label(value="🎭 イントネーション加工（選択的・有声区間のみ）",
          style={'font_weight': 'bold'}),
    HTML(value="<small style='color:#666'>※ 有声区間のみ加工し、無声音・休止は一切変えません。</small>"),
    intn_enable,
    intn_detail_box,
])

panel = VBox([
    header,
    VBox([
        VBox([
            Label(value="📂 入力ファイル", style={'font_weight': 'bold'}),
            source_dd, manual_path_box,
        ]),
        VBox([
            Label(value="🎭 声質プリセット", style={'font_weight': 'bold'}),
            preset_dd,
        ]),
        VBox([
            Label(value="🎚️ 手動調整", style={'font_weight': 'bold'}),
            HBox([pitch_sl,   pitch_lbl]),
            HBox([speed_sl,   speed_lbl]),
            HBox([formant_sl, formant_lbl]),
            HBox([robot_sl,   robot_lbl]),
        ]),
        intn_section,
        process_btn,
        proc_out,
    ], layout=widgets.Layout(padding='16px', gap='16px'))
], layout=widgets.Layout(
    border='2px solid #1a1a2e',
    border_radius='8px',
    margin='10px 0'
))

display(panel)